In [ ]:
pip install datasets transformers

## Import libraries

In [ ]:
from tqdm import tqdm
import pandas as pd
from typing import Optional, List, Tuple
from datasets import Dataset
from matplotlib import pyplot as plt

## Installing libraries in your environment

In [ ]:
import sys
print(sys.executable)
!{sys.executable} -m pip install bitsandbytes





## Dataset

Link: https://www.kaggle.com/datasets/chaitanyakck/medical-text

In [ ]:
!pip install langchain

## Loading and chunking dataset

![](https://miro.medium.com/v2/resize:fit:1127/1*Jq9bEbitg1Pv4oASwEQwJg.png)

In [ ]:
with open("train.txt", "r") as f:
    data = f.read()

In [ ]:
from langchain_core.documents import Document as LangchainDocument

RAW_KNOWLEDGE_BASE = LangchainDocument(page_content=data)


In [ ]:
MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # The maximum number of characters in a chunk: we selected this value arbitrarily
    chunk_overlap=100,  # The number of characters to overlap between chunks
    add_start_index=True,  # If `True`, includes chunk's start index in metadata
    strip_whitespace=True,  # If `True`, strips whitespace from the start and end of every document
    separators=MARKDOWN_SEPARATORS,
)

In [ ]:
docs_processed = text_splitter.split_documents([RAW_KNOWLEDGE_BASE])

In [ ]:
!pip install langchain_community
!pip install sentence-transformers

## Tokenizing/Vectorizing the dataset

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
EMBEDDING_MODEL_NAME = "thenlper/gte-small"

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    multi_process=True,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},  # Set `True` for cosine similarity
)

In [ ]:
emb = embedding_model.embed_query(docs_processed[0].page_content)

In [ ]:
import numpy as np
np.array(emb).shape

In [ ]:
!pip install pinecone-client

## Storing dataset into a vector database

Using: https://pinecone.com

In [ ]:
from tqdm.notebook import tqdm
from pinecone import Pinecone as pinecone

pc = pinecone(api_key="pcsk_7KQt8b_AD1bkpj8gyD67zh18EES2tBXXVapo43wBzzgVLuwntjiZ3FgFgcs6g77HfzKDHd")
index = pc.Index("lab-rag-index")

In [ ]:
upsert_data = []

for i, entry in tqdm(enumerate(docs_processed[:10])):
    text = entry.page_content
    vector = embedding_model.embed_query(text)
    upsert_data.append(
        {
            "id": "vec{}".format(i),
            "values": vector,
            "metadata": {"text": text}
        }
    )

In [ ]:
index.upsert( 
    vectors=upsert_data,
    namespace= "ns1"
)

## Loading a LLM

In [ ]:
!pip install ollama
from ollama import Client

OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "llama3:8b-instruct"

In [ ]:
ollama_client = Client(host=OLLAMA_HOST)
ollama_client.list()  # Quick connectivity check to local Ollama

In [ ]:
# If your local tag is different (for example, llama3.1:8b), update OLLAMA_MODEL above.
# Pull only if needed (can take time).
# ollama_client.pull(OLLAMA_MODEL)

In [ ]:
print(f"Using Ollama host: {OLLAMA_HOST}")
print(f"Using Ollama model: {OLLAMA_MODEL}")

In [ ]:
llm_healthcheck = ollama_client.chat(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": "Reply with OK"}],
)
print(llm_healthcheck["message"]["content"])

In [ ]:
def llm_model(prompt_text: str) -> str:
    response = ollama_client.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt_text}],
        options={"temperature": 0.4},
    )
    return response["message"]["content"]

In [ ]:
print(llm_model("Hey there!"))

## Prompting the model

In [ ]:
prompt = """
You are a helpful assistant that answers on medical questions based on the real information provided from different sources and in the context.
Give the rational and well written response. If you don't have proper info in the context, answer "I don't know"
Respond only to the question asked.

Context:
{}
---
Here is the question you need to answer.

Question: {}
"""

In [ ]:
print("Waiting for user input...")
user_input = "What are the symptoms of diabetes?"
print(f"Input received: {user_input}")

print("Embedding query...")
vectorized_input = embedding_model.embed_query(user_input)
print("Embedding complete")

print("Querying index...")
context = index.query(
    namespace="ns1",
    vector=vectorized_input,
    top_k=1,
    include_metadata=True
)
print("Query complete")

print("Generating answer...")
answer = llm_model(prompt.format(context['matches'][0]['metadata']['text'], user_input))
print("Generation complete")

print("AI response: ", answer)

In [ ]:
import sys
print("Python is working", file=sys.stderr)
sys.stderr.flush()

print("About to take input...")
sys.stdout.flush()

user_input = input("Type something: ")
print(f"You typed: {user_input}")

In [ ]:
context['matches'][0]['metadata']['text']